# Signed Jumps 02a — the perp price path after a jump

Protocol: `discussions/Signed_Jumps/003_claude_sj02a_preregistration.md` (fixed before this run).

SJ 01 jumps (unchanged engine, no refit; model (e) parameters from `01/01_fits.csv`) → cluster-starter events → signed perp returns at 1–60 min, immediate and delayed entry → rule chosen on DEV by net lower bound → verdict on TEST.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_events as se
for m in (sj,se):importlib.reload(m)
assert sj.VERSION=='SJ01-0.1.0' and se.VERSION=='SJ02a-0.1.0'
SYNTHETIC=os.getenv('SJ02_SYNTHETIC','0')=='1'
p=sj.Params();ep=se.EventParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj02-');OUT=Path(tmp.name)/'02'
    p=sj.Params(n_boot=200,half_lives=(1,3),min_test_jumps_per_sign=20);ep=se.EventParams(n_boot=300,min_test_events=50)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'02'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_events.py'),str(ROOT/'tests'/'test_signed_jumps.py')],
                       cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(se.registration(ep),indent=1))

## SJ 01 engine (unchanged) and fitted signal

In [ ]:
T0=time.time()
if SYNTHETIC:
    S={a:s for a,s in zip(p.assets,sj.simulate(n_days=50,n_assets=len(p.assets),eta=((0.2,0.05),(0.05,0.3)),half_life=1,seed=5,df=8,base_rate=1e-3).values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
assets,pan,dev_end=sj.prepare(S,p)
print('DEV/TEST boundary',dev_end,f'{time.time()-T0:.0f}s')
if SYNTHETIC:
    f0,_=sj.fit_ladder(assets,pan,p,models=('e',));rows=[]
    for (m,s),f in f0.items():
        r={'model':m,'sign':s,'half_life_min':f.half_life,'ll_dev':f.ll_dev,'n_dev':f.n_dev,'converged':f.converged}
        r.update(dict(zip(sj.design_names(pan,m,p),f.b)));r.update({f'eta{k}':v for k,v in enumerate(f.eta)});rows.append(r)
    tab=pd.DataFrame(rows)
else:
    tab=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_fits.csv')
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
fits=se.fits_from_table(tab,pan,p)
lam=se.intensities(fits,assets,pan)
ev=se.find_events(pan,lam,ep.cluster_gap)
print(f'{len(ev)} jumps, {ev.starter.sum()} cluster starters; pi_c quantiles:',ev.pi_c.quantile([.05,.2,.5,.8,.95]).round(3).tolist())

## Perp paths

In [ ]:
if SYNTHETIC:
    perp=se.plant_drift(S,assets,0.0,10)
else:
    perp={A:sj.load_spot_1m(BIN/f'{A}USDT_um_1m.parquet') for A in p.assets}
ev=se.attach_paths(ev,perp,ep)
cov=ev.groupby('asset')[[f'g_imm_{k}' for k in ep.horizons]].apply(lambda d:d.notna().mean()).round(3)
print('share of events with a perp path:');display(cov)
ev.to_csv(OUT/'02_events.csv.gz',index=False)
cuts=se.quintile_cuts(ev);print('pi_c DEV quintile cuts (20%, 80%):',np.round(cuts,4))

## DEV surface → rule; TEST verdict

In [ ]:
surf=pd.concat([se.surface(ev,ep,cuts,per,mode) for per in ('dev','test') for mode in ('imm','del')],ignore_index=True)
surf.to_csv(OUT/'02_surface.csv',index=False)
rule=se.select_rule(surf[(surf.period=='dev')&(surf.entry=='imm')],ep)
print('selected rule (DEV):',rule)
res_imm=se.evaluate(ev,rule,cuts,ep,'imm');res_del=se.evaluate(ev,rule,cuts,ep,'del')
evals=pd.DataFrame([{k:v for k,v in r.items() if k!='rule'} for r in (res_imm,res_del)])
display(evals.round(2));evals.to_csv(OUT/'02_test_eval.csv',index=False)
show=surf[(surf['filter']=='all')&(surf.direction=='continuation')].pivot_table(index='horizon_min',columns=['period','entry'],values='gross_bp')
print('Mean continuation return, all cluster starters (bp):');display(show.round(2))

## Shape of the move (TEST, descriptive)

In [ ]:
car=se.car_curve(ev,ep,cuts,'imm');car.to_csv(OUT/'02_car_curve.csv',index=False)
display(car.pivot_table(index=['sign','pi_c_quintile'],columns='horizon_min',values='mean_bp').round(2))
t=ev[ev.starter&~ev.dev]
exc=t.groupby(['asset','sign'])[['mfe_15','mae_15','jump_bp']].agg(['mean','median']).round(2)
display(exc);exc.to_csv(OUT/'02_excursions.csv')
allj=pd.concat([se.surface(ev.assign(starter=True),ep,cuts,'test','imm').assign(events='all_jumps')])
allj.to_csv(OUT/'02_surface_all_jumps.csv',index=False)
display(allj[(allj['filter']=='all')&(allj.direction=='continuation')][['horizon_min','gross_bp','gross_lo','gross_hi','n']].round(2))

## Decision

In [ ]:
dec={'version':se.VERSION,'status':ep.status,'rule':rule,'verdict':res_imm['verdict'],
     'test_gross_bp':res_imm['gross_bp'],'test_gross_lo':res_imm['gross_lo'],
     'test_net_12bp':res_imm['net_12bp'],'test_net_12bp_lo':res_imm['net_12bp_lo'],
     'test_net_7bp':res_imm['net_7bp'],'test_net_7bp_lo':res_imm['net_7bp_lo'],
     'delayed_entry':{k:res_del[k] for k in ('gross_bp','gross_lo','net_12bp','net_12bp_lo','verdict')},
     'test_events':res_imm['test_n'],'build_SJ02b':res_imm['verdict']=='NET_EDGE','synthetic':SYNTHETIC}
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'02_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'02_manifest.json').write_text(json.dumps({'registration':se.registration(ep),'runtime_s':time.time()-T0,
        'events':int(len(ev)),'starters':int(ev.starter.sum())},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')